In [1]:
from data_model_loader import *

model = load_model()
annotations, images = load_coco_2014_dataset()

# load file paths
with open('config.json', 'r') as file:
    config = json.load(file)

coco_folder = config['coco_folder']
annotation_file_path = config['annotation_file_path']

# load image ids
with open(annotation_file_path, 'r') as f:
        coco_data = json.load(f)

filename_to_image_id= {image['file_name']: image["id"] for image in coco_data['images']}


OD model already present locally.
data/coco2014\val2014.zip already exists. Skipping download.
data/coco2014\val2014 already exists. Skipping extraction.
data/coco2014\annotations_trainval2014.zip already exists. Skipping download.
data/coco2014\annotations_trainval2014 already exists. Skipping extraction.
COCO 2014 dataset download and extraction complete.
Get Annotations ..
Randomly selecting 1500 pictures ..
Done!


In [ ]:
from PIL import Image
import torchvision.transforms as transforms
import pathlib
from tqdm import tqdm


def predict(model, data_path, images, n_images=5):
    data_path = pathlib.Path(data_path)
    results = []

    for img in tqdm(images[:n_images], desc="Loading images"):
        # load image
        image_path = data_path/"val2014"/"val2014"/img
        image = Image.open(image_path)
        width, height = image.size

        # convert greyscale to rgb
        if image.mode == 'L':
            image = image.convert('RGB')

        # transform to tensor
        transform = transforms.Compose([
            transforms.ToTensor()  # Konvertiert das Bild zu einem Tensor [C, H, W] mit Werten zwischen 0 und 1
        ])
        
        image_tensor = transform(image)
        
        image_tensor = (image_tensor * 255).byte()  # convert to uint8
        image_tensor = image_tensor.permute(1, 2, 0)  # [C, H, W] -> [H, W, C]
        image_tensor = image_tensor.unsqueeze(0)

        # inference
        detector_output = model(image_tensor)
              
        # evaluate
        n_detections = int(detector_output["num_detections"].numpy()[0])

        for i in range(n_detections):
            ymin, xmin, ymax, xmax = detector_output["detection_boxes"].numpy()[0][i]
            ymin = ymin * height
            ymax = ymax * height
            xmin = xmin * width
            xmax = xmax * width

            result = {
                    "image_id" : int(filename_to_image_id[img]),
                    "category_id":int(detector_output["detection_classes"].numpy()[0][i]),
                    "bbox": [xmin, ymin, xmax - xmin, ymax - ymin], # detector_output["detection_boxes"].numpy()[0][i].tolist(), # needs to be list
                    "score": float(detector_output["detection_scores"].numpy()[0][i])
            }
            results.append(result)
        

    return results
    
def get_annotation(annotation_file_path):
    with open(annotation_file_path, 'r') as f:
        coco_data = json.load(f)
    category_map = {category['id']: category['name'] for category in coco_data['categories']}
    return category_map
def preprocess_image(image_tensor):
    # Check if the image is grayscale (i.e., has only 1 channel)
    if image_tensor.shape[-1] == 1:
        # Convert grayscale to RGB by replicating the single channel three times
        image_tensor = tf.image.grayscale_to_rgb(image_tensor)
    return image_tensor


In [9]:
results = predict(model, coco_folder, images, n_images=len(images))
results

Loading images: 100%|██████████| 1500/1500 [00:40<00:00, 36.65it/s]


[{'image_id': 99230,
  'category_id': 17,
  'bbox': [292.5440216064453,
   2.643885612487793,
   347.4559783935547,
   472.18178272247314],
  'score': 0.6335581541061401},
 {'image_id': 99230,
  'category_id': 75,
  'bbox': [140.5400562286377,
   199.8222541809082,
   238.98741722106934,
   99.6946907043457],
  'score': 0.5928142666816711},
 {'image_id': 99230,
  'category_id': 63,
  'bbox': [172.94687271118164,
   3.864741325378418,
   467.05312728881836,
   467.24714756011963],
  'score': 0.5526977181434631},
 {'image_id': 99230,
  'category_id': 84,
  'bbox': [149.82457160949707,
   197.41172790527344,
   244.2764949798584,
   101.76246643066406],
  'score': 0.4902001619338989},
 {'image_id': 99230,
  'category_id': 63,
  'bbox': [0.0, 5.527253150939941, 350.0386047363281, 468.3670949935913],
  'score': 0.4584502577781677},
 {'image_id': 99230,
  'category_id': 75,
  'bbox': [142.36860275268555,
   198.47684383392334,
   133.8513946533203,
   86.0047960281372],
  'score': 0.45266199

In [331]:
results = predict(model, coco_folder, images, n_images=100)
results

Loading images: 100%|██████████| 100/100 [00:03<00:00, 29.66it/s]


[{'image_id': 408955,
  'category_id': 63,
  'bbox': [12.850456237792969,
   202.7556037902832,
   285.46220779418945,
   153.6110544204712],
  'score': 0.7492043375968933},
 {'image_id': 408955,
  'category_id': 63,
  'bbox': [113.88839721679688,
   143.82673144340515,
   203.33446502685547,
   139.652978181839],
  'score': 0.7428119778633118},
 {'image_id': 408955,
  'category_id': 63,
  'bbox': [45.47842025756836,
   159.34141159057617,
   253.24443817138672,
   155.39435863494873],
  'score': 0.6123853921890259},
 {'image_id': 408955,
  'category_id': 84,
  'bbox': [269.03709411621094,
   207.56495475769043,
   45.003013610839844,
   24.87553596496582],
  'score': 0.5190582871437073},
 {'image_id': 408955,
  'category_id': 67,
  'bbox': [245.47584533691406,
   217.41546392440796,
   196.8291473388672,
   111.60272598266602],
  'score': 0.5053174495697021},
 {'image_id': 408955,
  'category_id': 72,
  'bbox': [532.7688217163086,
   148.61289739608765,
   99.21485900878906,
   209.84

In [332]:
# create result dir
results_dir = pathlib.Path('results/')
results_dir.mkdir(exist_ok=True, parents=True)
results_file = results_dir/"results.json"

# store results
with open(results_file, 'w') as json_file:
    json.dump(results, json_file, indent=4)

In [333]:
%matplotlib inline
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import numpy as np
import skimage.io as io
import pylab
pylab.rcParams['figure.figsize'] = (10.0, 8.0)

annType = ['segm','bbox','keypoints']
annType = annType[1]      #specify type here
prefix = 'person_keypoints' if annType=='keypoints' else 'instances'
annType

'bbox'

In [334]:
annotation_file_path

'data/coco2014/annotations_trainval2014/annotations/instances_val2014.json'

In [335]:
#initialize COCO ground truth api
cocoGt=COCO(annotation_file_path)

loading annotations into memory...
Done (t=3.38s)
creating index...
index created!


In [336]:
#cocoGt.

In [337]:
#initialize COCO detections api
dataDir='../'
dataType='val2014'


resFile='%s/results/%s_%s_fake%s100_results.json'
resFile = resFile%(dataDir, prefix, dataType, annType)
cocoDt=cocoGt.loadRes('results/results.json')

Loading and preparing results...
DONE (t=0.06s)
creating index...
index created!


In [338]:
cocoGt.getImgIds()

[391895,
 522418,
 184613,
 318219,
 554625,
 397133,
 574769,
 60623,
 309022,
 5802,
 222564,
 118113,
 193271,
 224736,
 483108,
 403013,
 374628,
 328757,
 384213,
 293802,
 86408,
 37777,
 372938,
 386164,
 223648,
 204805,
 113588,
 384553,
 337264,
 368402,
 12448,
 252219,
 79841,
 87038,
 174482,
 515289,
 562150,
 542145,
 412151,
 403385,
 579003,
 540186,
 242611,
 51191,
 269105,
 294832,
 462565,
 144941,
 173350,
 60760,
 324266,
 166532,
 262284,
 360772,
 6818,
 191381,
 111076,
 340559,
 258985,
 509822,
 321107,
 229643,
 125059,
 455483,
 436141,
 129001,
 232262,
 61181,
 166323,
 580041,
 326781,
 387362,
 138079,
 556616,
 472621,
 192440,
 86320,
 256668,
 383445,
 565797,
 81922,
 50125,
 364521,
 394892,
 1146,
 310391,
 97434,
 463836,
 241876,
 156832,
 480985,
 458054,
 270721,
 462341,
 310103,
 32992,
 122851,
 540763,
 331352,
 138246,
 197254,
 32907,
 251252,
 37675,
 159537,
 268556,
 271177,
 75051,
 549399,
 85160,
 559665,
 296649,
 19358,
 459912,

In [339]:
imgIds=sorted(cocoGt.getImgIds())
imgIds=imgIds[0:5000]
imgId = imgIds[np.random.randint(100)]

In [340]:
cocoEval = COCOeval(cocoGt,cocoDt, 'bbox')
cocoEval.params.imgIds = imgIds
cocoEval.evaluate()
cocoEval.accumulate()
cocoEval.summarize()

Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=3.88s).
Accumulating evaluation results...
DONE (t=0.88s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.002
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.002
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.002
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.001
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.002
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.001
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.001
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.001
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets=100

### Old

In [341]:
classes = get_annotation(annotation_file_path)

In [342]:
classes

{1: 'person',
 2: 'bicycle',
 3: 'car',
 4: 'motorcycle',
 5: 'airplane',
 6: 'bus',
 7: 'train',
 8: 'truck',
 9: 'boat',
 10: 'traffic light',
 11: 'fire hydrant',
 13: 'stop sign',
 14: 'parking meter',
 15: 'bench',
 16: 'bird',
 17: 'cat',
 18: 'dog',
 19: 'horse',
 20: 'sheep',
 21: 'cow',
 22: 'elephant',
 23: 'bear',
 24: 'zebra',
 25: 'giraffe',
 27: 'backpack',
 28: 'umbrella',
 31: 'handbag',
 32: 'tie',
 33: 'suitcase',
 34: 'frisbee',
 35: 'skis',
 36: 'snowboard',
 37: 'sports ball',
 38: 'kite',
 39: 'baseball bat',
 40: 'baseball glove',
 41: 'skateboard',
 42: 'surfboard',
 43: 'tennis racket',
 44: 'bottle',
 46: 'wine glass',
 47: 'cup',
 48: 'fork',
 49: 'knife',
 50: 'spoon',
 51: 'bowl',
 52: 'banana',
 53: 'apple',
 54: 'sandwich',
 55: 'orange',
 56: 'broccoli',
 57: 'carrot',
 58: 'hot dog',
 59: 'pizza',
 60: 'donut',
 61: 'cake',
 62: 'chair',
 63: 'couch',
 64: 'potted plant',
 65: 'bed',
 67: 'dining table',
 70: 'toilet',
 72: 'tv',
 73: 'laptop',
 74: 'mo

In [343]:
obj = results[images[0]]["detector_output"]
obj.keys()

TypeError: list indices must be integers or slices, not str

In [ ]:
n_detections = int(obj["num_detections"].numpy()[0])
n_detections

100

In [ ]:
result = []
n_detections = int(obj["num_detections"].numpy()[0])

for i in range(n_detections):
     r = [
        {"category_id":obj["detection_classes"].numpy()[0][i],
         "score": obj["detection_scores"].numpy()[0][i],
         "bbox": obj["detection_boxes"].numpy()[0][i]}
     ]
     result.append(r)

result[2]

[{'category_id': 16.0,
  'score': 0.3826751,
  'bbox': array([0.1492896 , 0.4810276 , 0.31278604, 0.837199  ], dtype=float32)}]

In [ ]:
with open(annotation_file_path, 'r') as f:
        coco_data = json.load(f)

In [ ]:
filename_to_image_id['COCO_val2014_000000391895.jpg']

391895

In [ ]:
filename_to_image_id= {image['file_name']: image["id"] for image in coco_data['images']}
filename_to_image_id

{'COCO_val2014_000000391895.jpg': 391895,
 'COCO_val2014_000000522418.jpg': 522418,
 'COCO_val2014_000000184613.jpg': 184613,
 'COCO_val2014_000000318219.jpg': 318219,
 'COCO_val2014_000000554625.jpg': 554625,
 'COCO_val2014_000000397133.jpg': 397133,
 'COCO_val2014_000000574769.jpg': 574769,
 'COCO_val2014_000000060623.jpg': 60623,
 'COCO_val2014_000000309022.jpg': 309022,
 'COCO_val2014_000000005802.jpg': 5802,
 'COCO_val2014_000000222564.jpg': 222564,
 'COCO_val2014_000000118113.jpg': 118113,
 'COCO_val2014_000000193271.jpg': 193271,
 'COCO_val2014_000000224736.jpg': 224736,
 'COCO_val2014_000000483108.jpg': 483108,
 'COCO_val2014_000000403013.jpg': 403013,
 'COCO_val2014_000000374628.jpg': 374628,
 'COCO_val2014_000000328757.jpg': 328757,
 'COCO_val2014_000000384213.jpg': 384213,
 'COCO_val2014_000000293802.jpg': 293802,
 'COCO_val2014_000000086408.jpg': 86408,
 'COCO_val2014_000000037777.jpg': 37777,
 'COCO_val2014_000000372938.jpg': 372938,
 'COCO_val2014_000000386164.jpg': 38616

In [ ]:
coco_data["images"]

[{'license': 3,
  'file_name': 'COCO_val2014_000000391895.jpg',
  'coco_url': 'http://images.cocodataset.org/val2014/COCO_val2014_000000391895.jpg',
  'height': 360,
  'width': 640,
  'date_captured': '2013-11-14 11:18:45',
  'flickr_url': 'http://farm9.staticflickr.com/8186/8119368305_4e622c8349_z.jpg',
  'id': 391895},
 {'license': 4,
  'file_name': 'COCO_val2014_000000522418.jpg',
  'coco_url': 'http://images.cocodataset.org/val2014/COCO_val2014_000000522418.jpg',
  'height': 480,
  'width': 640,
  'date_captured': '2013-11-14 11:38:44',
  'flickr_url': 'http://farm1.staticflickr.com/1/127244861_ab0c0381e7_z.jpg',
  'id': 522418},
 {'license': 3,
  'file_name': 'COCO_val2014_000000184613.jpg',
  'coco_url': 'http://images.cocodataset.org/val2014/COCO_val2014_000000184613.jpg',
  'height': 336,
  'width': 500,
  'date_captured': '2013-11-14 12:36:29',
  'flickr_url': 'http://farm3.staticflickr.com/2169/2118578392_1193aa04a0_z.jpg',
  'id': 184613},
 {'license': 3,
  'file_name': 'COC